In [ ]:
## setup

import sys, os, time, gc
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # so `src` is importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import *
from src.data_loader import find_files, pre_cache_all, build_area_series, load_daily
from src.preprocessing import (make_windows, train_val_test_split,
                                normalise, make_xgb_features, add_time_features)
from src.metrics import evaluate, mae, rmse, mape
from src.models.sarima_model import SarimaForecaster
from src.models.lstm_model  import LSTMForecaster
from src.models.xgboost_model import XGBoostForecaster
from src.evaluation import (walk_forward_sarima, walk_forward_dl,
                            walk_forward_xgboost)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)

In [ ]:
## top-3 areas by total traffic

AGG_CACHE = CACHE_DIR / "total_traffic_per_area.parquet"
assert AGG_CACHE.exists(), "Run Task 2.1 first to build the aggregation cache."

total_traffic = (pd.read_parquet(AGG_CACHE)
                    .set_index("Square id")["total_traffic"]
                    .sort_values(ascending=False))
top3 = total_traffic.head(3).index.tolist()
print("Top-3 areas:", top3)

NameError: name 'CACHE_DIR' is not defined

In [ ]:
## Building times series for the top-3 areas

area_series = {}
for sid in top3:
    s = build_area_series(sid)
    area_series[sid] = s
    print(f"Square {sid}: {len(s):,} intervals | "
            f"{s.index.min()} → {s.index.max()} | mean={s.mean():.2f}")

In [ ]:
# train/val/test split

splits = {}
for sid, s in area_series.items():
    tr, va, te = train_val_test_split(
        s,
        train_end=TRAIN_END,
        val_start=VAL_START, val_end=VAL_END,
        test_start=TEST_START, test_end=TEST_END,
    )
    splits[sid] = {"train": tr, "val": va, "test": te}
    print(f"Square {sid}: train={len(tr)} val={len(va)} test={len(te)}")

In [ ]:
## log experiment

experiment_log = []   # every row = one experiment

def log_experiment(model_name, area, params, metrics,
                    train_time=None, exec_time=None, notes=""):
    experiment_log.append({
        "model": model_name, "area": area, "params": params,
        "MAE": metrics["MAE"], "RMSE": metrics["RMSE"], "MAPE": metrics["MAPE"],
        "train_time_s": train_time, "exec_time_s": exec_time,
        "notes": notes,
    })
    print(f"[{model_name}] area={area} | MAE={metrics['MAE']:.3f} "
            f"RMSE={metrics['RMSE']:.3f} MAPE={metrics['MAPE']:.2f}%"
            + (f" | train={train_time:.1f}s" if train_time else "")
            + (f" | exec={exec_time:.1f}s" if exec_time else ""))

In [ ]:
## sarima experiments

highest = top3[0]
tr, va, te = (splits[highest]["train"],
                splits[highest]["val"],
                splits[highest]["test"])

# --- Round 1: baseline (1,0,1)(1,0,1,144) ---
base = SarimaForecaster(order=(1,0,1), seasonal_order=(1,0,1,144))
base.fit(tr)
val_pred = base.model_fit.forecast(steps=len(va))
m = evaluate(va.values, np.asarray(val_pred))
log_experiment("SARIMA", highest, base.order, m, train_time=base.train_time_,
                notes="baseline (1,0,1)(1,0,1,144)")

# --- Round 2: mini grid search ---
gs = SarimaForecaster.grid_search(
    tr, va,
    p_range=(0,1,2), d_range=(0,1), q_range=(0,1),
    P_range=(0,1), D_range=(0,1), Q_range=(0,1),
    s=144, max_evals=16,
)
print(gs)

# --- Round 3: refit best config and evaluate ---
best = gs.iloc[0]
best_order = eval(best["order"]) if isinstance(best["order"], str) else best["order"]
best_seasonal = eval(best["seasonal"]) if isinstance(best["seasonal"], str) else best["seasonal"]

sarima_best = SarimaForecaster(order=best_order, seasonal_order=best_seasonal)
sarima_best.fit(tr)
val_pred = sarima_best.model_fit.forecast(steps=len(va))
m = evaluate(va.values, np.asarray(val_pred))
log_experiment("SARIMA", highest, best_order, m,
                train_time=sarima_best.train_time_,
                notes=f"best grid config {best_seasonal}")

In [ ]:
## lstm experiments

# Normalise using train statistics only
(tr_n, va_n, te_n), (mu, sd) = normalise(tr, va, te)

X_tr, y_tr = make_windows(tr_n, SEQ_LEN)
X_va, y_va = make_windows(va_n, SEQ_LEN)
X_te, y_te = make_windows(te_n, SEQ_LEN)

# --- Round 1: baseline single-layer LSTM(64) ---
cfg1 = dict(seq_len=SEQ_LEN, hidden_units=64, n_layers=1,
            dropout=0.0, learning_rate=1e-3, batch_size=64, epochs=20)
m1 = LSTMForecaster(**cfg1)
m1.fit(X_tr, y_tr, X_va, y_va)
yp1 = m1.predict(X_va)
metrics1 = evaluate(y_va, yp1)
log_experiment("LSTM", highest, cfg1, metrics1, train_time=m1.train_time_,
                notes="baseline 1×LSTM(64)")

# --- Round 2: try dropout + larger hidden ---
cfg2 = dict(seq_len=SEQ_LEN, hidden_units=128, n_layers=1,
            dropout=0.2, learning_rate=1e-3, batch_size=64, epochs=20)
m2 = LSTMForecaster(**cfg2)
m2.fit(X_tr, y_tr, X_va, y_va)
metrics2 = evaluate(y_va, m2.predict(X_va))
log_experiment("LSTM", highest, cfg2, metrics2, train_time=m2.train_time_,
                notes="wider + dropout")

# --- Round 3: 2-layer stack + early stopping ---
cfg3 = dict(seq_len=SEQ_LEN, hidden_units=64, n_layers=2,
            dropout=0.2, learning_rate=5e-4, batch_size=64,
            epochs=30, patience=5)
m3 = LSTMForecaster(**cfg3)
m3.fit(X_tr, y_tr, X_va, y_va)
metrics3 = evaluate(y_va, m3.predict(X_va))
log_experiment("LSTM", highest, cfg3, metrics3, train_time=m3.train_time_,
                notes="2-layer stack, lower LR")

# --- Pick best LSTM config by validation MAE ---
best_cfg = min(
    [("cfg1", cfg1, metrics1, m1),
        ("cfg2", cfg2, metrics2, m2),
        ("cfg3", cfg3, metrics3, m3)],
    key=lambda x: x[2]["MAE"],
)
print("Best LSTM config:", best_cfg[0], "| val MAE =", best_cfg[2]["MAE"])
lstm_best = best_cfg[3]

In [ ]:
## xgboost experiments

xgb_tr = make_xgb_features(tr)
xgb_va = make_xgb_features(pd.concat([tr, va]))
xgb_va = xgb_va.iloc[len(xgb_tr):]

# --- Round 1: shallow trees ---
x1 = XGBoostForecaster(n_estimators=300, max_depth=4, learning_rate=0.05)
x1.fit(xgb_tr, val_df=xgb_va)
metrics_x1 = evaluate(xgb_va["y"].values, x1.predict(xgb_va))
log_experiment("XGBoost", highest, x1.params, metrics_x1,
                train_time=x1.train_time_, notes="depth=4, n=300")

# --- Round 2: deeper, more trees ---
x2 = XGBoostForecaster(n_estimators=600, max_depth=6, learning_rate=0.03,
                        subsample=0.8, colsample_bytree=0.8)
x2.fit(xgb_tr, val_df=xgb_va)
metrics_x2 = evaluate(xgb_va["y"].values, x2.predict(xgb_va))
log_experiment("XGBoost", highest, x2.params, metrics_x2,
                train_time=x2.train_time_, notes="depth=6, n=600, lr=0.03")

# --- Round 3: regularised ---
x3 = XGBoostForecaster(n_estimators=600, max_depth=6, learning_rate=0.03,
                        subsample=0.8, colsample_bytree=0.8,
                        reg_alpha=0.1, reg_lambda=2.0)
x3.fit(xgb_tr, val_df=xgb_va)
metrics_x3 = evaluate(xgb_va["y"].values, x3.predict(xgb_va))
log_experiment("XGBoost", highest, x3.params, metrics_x3,
                train_time=x3.train_time_, notes="+ L1/L2 regularisation")

# --- Best XGB config ---
best_xgb = min([(x1, metrics_x1), (x2, metrics_x2), (x3, metrics_x3)],
                key=lambda t: t[1]["MAE"])[0]

In [ ]:
## Evaluation on Test Week

def run_full_evaluation(model_name, model, sid):
    tr = splits[sid]["train"]
    va = splits[sid]["val"]
    te = splits[sid]["test"]
    hist = pd.concat([tr, va])

    if model_name == "SARIMA":
        yp, exec_t = walk_forward_sarima(model, hist, te)
        yp_real = yp
    elif model_name == "LSTM":
        (_, _, te_n), (mu, sd) = normalise(tr, va, te)
        hist_n = (hist - mu) / sd
        yp_n, exec_t = walk_forward_dl(model, hist_n.values, te_n.values, SEQ_LEN)
        yp_real = yp_n * sd + mu
    elif model_name == "XGBoost":
        yp_real, exec_t = walk_forward_xgboost(
            model, hist, te, make_xgb_features)
    metrics = evaluate(te.values, yp_real)
    return metrics, yp_real, exec_t


results = {}
predictions = {}
for sid in top3:
    results[sid] = {}
    predictions[sid] = {}
    for name, model in [("SARIMA", sarima_best),
                            ("LSTM", lstm_best),
                            ("XGBoost", best_xgb)]:
        m, yp, exec_t = run_full_evaluation(name, model, sid)
        results[sid][name] = m
        predictions[sid][name] = yp
        log_experiment(name, sid, "final", m,
                        train_time=getattr(model, "train_time_", None),
                        exec_time=exec_t, notes="test-week walk-forward")

In [ ]:
## plots

import matplotlib.dates as mdates

for sid in top3:
    te = splits[sid]["test"]
    for name in ["SARIMA", "LSTM", "XGBoost"]:
        yp = predictions[sid][name]
        fig, ax = plt.subplots(figsize=(14, 4))
        ax.plot(te.index, te.values, label="Actual", linewidth=1.2,
                color="black")
        ax.plot(te.index, yp, label=f"{name} prediction",
                linewidth=1.0, alpha=0.85)
        ax.set_title(f"Square {sid} — {name} | Week Dec 16–22")
        ax.set_ylabel("Internet traffic")
        ax.set_xlabel("Date")
        ax.legend()
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
        plt.tight_layout()
        fname = FIG_DIR / f"forecast_{name.lower()}_sq{sid}.png"
        plt.savefig(fname, dpi=150, bbox_inches="tight")
        plt.show()

In [ ]:
## table of metrics

all_tables = {}
for sid in top3:
    df_metrics = pd.DataFrame(results[sid]).T
    df_metrics = df_metrics[["MAE", "MAPE", "RMSE"]].round(4)
    df_metrics.index.name = "Model"
    all_tables[sid] = df_metrics
    print(f"\n=== Square {sid} — Test metrics ===")
    display(df_metrics)
    df_metrics.to_csv(MET_DIR / f"metrics_sq{sid}.csv")

In [ ]:
## training and execution times table

timing_rows = []
for name in ["SARIMA", "LSTM", "XGBoost"]:
    train_ts = [getattr(m, "train_time_", None)
                for m in {"SARIMA": sarima_best,
                            "LSTM": lstm_best,
                            "XGBoost": best_xgb}[name].__class__.__dict__.values()
                if hasattr(m, "train_time_")]  # placeholder
    timing_rows.append({"Model": name, "Hardware": "See README"})

# Simpler: read from experiment_log
log_df = pd.DataFrame(experiment_log)
final = log_df[log_df["notes"] == "test-week walk-forward"]
summary_time = (final.groupby("model")[["train_time_s", "exec_time_s"]]
                        .agg(["mean", "std"]).round(3))
display(summary_time)
summary_time.to_csv(MET_DIR / "timing_summary.csv")

In [ ]:
## analysis

# Identify the worst 1-hour window per model per area
def worst_window(y_true, y_pred, window=6):
    errs = np.abs(np.asarray(y_true) - np.asarray(y_pred))
    rm = pd.Series(errs).rolling(window).mean()
    idx = int(rm.idxmax())
    return idx - window + 1, idx

for sid in top3:
    te = splits[sid]["test"]
    print(f"\n=== Square {sid} — worst 1-hour windows ===")
    for name in ["SARIMA", "LSTM", "XGBoost"]:
        a, b = worst_window(te.values, predictions[sid][name])
        print(f"  {name}: indices {a}–{b} "
                f"({te.index[a]} → {te.index[b]}) | "
                f"max 1h MAE = "
                f"{np.mean(np.abs(te.values[a:b+1]-predictions[sid][name][a:b+1])):.3f}")

In [ ]:
## log & predictions

log_df = pd.DataFrame(experiment_log)
log_df.to_csv(MET_DIR / "experiment_log.csv", index=False)

for sid in top3:
    for name, yp in predictions[sid].items():
        pd.DataFrame({"datetime": splits[sid]["test"].index,
                        "actual": splits[sid]["test"].values,
                        "pred": yp}
        ).to_csv(PRED_DIR / f"pred_sq{sid}_{name.lower()}.csv", index=False)

print("Saved experiment log and predictions.")